# MedTrace FL — Federated Learning for Privacy-Preserving Medical AI

**Kaggle edition** — pure entry point into the GitHub repository.  
All training logic lives in the repo. Nothing is duplicated here.

| What runs | Where it lives |
|---|---|
| `run_simulation()` | `src/fl_simulate.py` |
| `FLConfig` dataclasses | `src/fl_config.py` |
| `AdaptiveDPMechanism` | `src/fl_adaptive_dp.py` |
| Evaluation + plots | `src/fl_evaluator.py`, `src/fl_plots.py` |

**Auto-resume:** checkpoints are saved to `/kaggle/working/` which persists  
between runs. If the session times out, click **Run All** — training resumes  
from the last completed round automatically.

---
## Before running — 3 required settings

On the right sidebar:
1. **Accelerator** → select **GPU T4 x1** (or P100)
2. **Internet** → turn **On** (needed to clone from GitHub)
3. Click **Save** then **Run All**

## Kaggle free GPU budget
| GPU | VRAM | Hours/week | 20 rounds (est.) |
|-----|------|-----------|------------------|
| T4  | 16 GB | 30 hrs ✅ | ~5–7 hrs ✅ |
| P100 | 16 GB | 30 hrs ✅ | ~4–6 hrs ✅ |

> Both GPUs are free and have enough VRAM. T4 is more commonly available.


## Step 1 — Verify GPU


In [ ]:
import os, sys, torch

# ── GPU check ─────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected.\n'
        'On the right sidebar: Accelerator → GPU T4 x1, then Run All again.'
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu_name}')
print(f'VRAM : {gpu_mem:.1f} GB')

# ── Kaggle working directory (persists between runs — no auth needed) ─────
WORK_DIR = '/kaggle/working/medtrace'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Work : {WORK_DIR}')
print('Ready.')


## Step 2 — Clone repository and install dependencies

> **If you see a network error here:** Internet is not enabled.  
> Go to the right sidebar → Internet → On → Save, then Run All again.


In [ ]:
import subprocess, importlib

REPO_URL = 'https://github.com/vivek797029/MEDTRACE.git'
REPO_DIR = '/kaggle/working/MEDTRACE'
SRC_DIR  = '/kaggle/working/MEDTRACE/src'

# ── Clone or pull ─────────────────────────────────────────────────────────
if os.path.exists(REPO_DIR):
    print('Repository already present — pulling latest...')
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or 'Already up to date.')
else:
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
        check=True
    )
    print('Clone complete.')

# ── Install from requirements.txt ─────────────────────────────────────────
req = os.path.join(REPO_DIR, 'requirements.txt')
print('\nInstalling dependencies (~2 min on first run)...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', req],
    check=True
)
print('Install complete.')

# ── Add src/ to Python path ───────────────────────────────────────────────
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# ── Verify all project modules are importable ─────────────────────────────
print('\nModule check:')
for mod_name in [
    'fl_config', 'fl_simulate', 'fl_adaptive_dp',
    'fl_evaluator', 'fl_plots', 'fl_tracker',
]:
    importlib.import_module(mod_name)
    print(f'  OK  {mod_name}')

print('\nSetup complete — ready to train.')


## Step 3 — Configure the experiment

All paths point to `/kaggle/working/` which Kaggle preserves between runs.  
Edit `fl_rounds` or `hospitals` to change the experiment size.


In [ ]:
import math
from fl_config import (
    FLConfig, DPConfig, AdaptiveDPConfig,
    LoRAConfig, TrainingConfig, EvalConfig,
    TrackerConfig, HospitalRegistry,
)

# ── Paths (all inside /kaggle/working/ — persists across runs) ───────────
OUTPUT_DIR  = os.path.join(WORK_DIR, 'outputs')
CKPT_DIR    = os.path.join(WORK_DIR, 'checkpoints')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# ── Build experiment config ───────────────────────────────────────────────
cfg = FLConfig(
    # Model
    base_model   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',

    # Federated learning
    fl_rounds    = 20,      # set to 2 for a quick smoke-test
    local_epochs = 1,
    hospitals    = HospitalRegistry.build(3),   # 3 specialty hospitals

    # LoRA adapter
    lora = LoRAConfig(r=8, alpha=16, dropout=0.05),

    # Differential privacy (epsilon, delta)-DP via Gaussian mechanism
    dp = DPConfig(
        enabled       = True,
        epsilon       = 8.0,
        delta         = 1e-5,
        max_grad_norm = 1.0,
    ),

    # Adaptive per-client DP (novel contribution)
    adaptive_dp = AdaptiveDPConfig(
        enabled              = True,
        ema_alpha            = 0.1,
        min_epsilon_fraction = 0.1,
    ),

    # Training
    training = TrainingConfig(
        batch_size                  = 4,
        gradient_accumulation_steps = 4,
        learning_rate               = 2e-4,
        max_length                  = 512,
    ),

    # Evaluation every 5 rounds
    eval = EvalConfig(
        enabled             = True,
        eval_every_n_rounds = 5,
        num_eval_samples    = 200,
    ),

    # Tracking: 'none' | 'mlflow' | 'wandb'
    tracker = TrackerConfig(backend='none'),

    # All 4 path fields — must be set explicitly so they point to /kaggle/working/
    output_dir          = OUTPUT_DIR,
    global_model_dir    = os.path.join(OUTPUT_DIR, 'global_model'),
    hospital_models_dir = os.path.join(OUTPUT_DIR, 'hospital_models'),
    metrics_dir         = os.path.join(OUTPUT_DIR, 'metrics'),
)

# ── Summary ───────────────────────────────────────────────────────────────
print('Experiment configuration')
print(f'  Model       : {cfg.base_model}')
print(f'  Hospitals   : {cfg.num_hospitals}')
for hid, h in cfg.hospitals.items():
    steps = h.num_samples // cfg.training.batch_size
    print(f'    {hid}: {h.name} — {h.num_samples} samples, {steps} steps/round')
total_steps = sum(
    h.num_samples // cfg.training.batch_size
    for h in cfg.hospitals.values()
) * cfg.fl_rounds * cfg.local_epochs
print(f'  FL rounds   : {cfg.fl_rounds}')
print(f'  Total steps : ~{total_steps:,}')
print(f'  Est. time   : ~{total_steps * 1.0 / 3600:.1f}–{total_steps * 1.2 / 3600:.1f} hrs on T4/P100')
if cfg.dp.enabled:
    print(f'  DP          : enabled  epsilon={cfg.dp.epsilon}  sigma={cfg.dp.sigma:.4f}')
    print(f'  Adaptive DP : {cfg.adaptive_dp.enabled}')
print(f'  Output dir  : {cfg.output_dir}')
print(f'  Checkpoints : {CKPT_DIR}')


## Step 4 — Run federated training

One call to `run_simulation()`. All logic runs from the cloned repository.  
**Session timed out?** Click Run All — resumes from the last checkpoint in `/kaggle/working/`.

**Training progress** — you will see one INFO block per round:  
```
INFO | fl_server | Aggregating Round 1 (3 clients)...
INFO | fl_client | hospital_00 | Round 1 | Loss: 1.84
INFO | fl_server | Aggregation complete | Avg loss: 1.71 | Round: 1/20
```
Each completed round = 5% of total training done.


In [ ]:
from fl_simulate import run_simulation, setup_logging
import logging

setup_logging(logging.INFO)

report = run_simulation(
    cfg,
    checkpoint_dir = CKPT_DIR,
)

# ── Summary ───────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('Training complete')
print(f'  Time    : {report["total_training_time"] / 60:.1f} min')
print(f'  Rounds  : {len(report["round_metrics"])}')
if report['round_metrics']:
    last = report['round_metrics'][-1]
    print(f'  Loss    : {last["avg_loss"]:.4f}')
    print(f'  Diverge : {last["weight_divergence"]:.6f}')
if 'adaptive_dp_summary' in report:
    print('  DP budget per hospital:')
    for hid, s in report['adaptive_dp_summary']['per_client'].items():
        pct = s['budget_spent'] / cfg.dp.epsilon * 100
        print(f'    {hid}: {s["budget_spent"]:.3f} / {cfg.dp.epsilon}  ({pct:.1f}%)')
print('=' * 60)


## Step 5 — Plot training curves

Saved to `/kaggle/working/medtrace/outputs/plots/`.  
Download them from the Output tab on the right.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from fl_evaluator import EvalAccumulator, EvalResult
from fl_plots    import ResultsPlotter

# ── Build accumulator from training report ────────────────────────────────
acc = EvalAccumulator()
acc.set_metadata(**{k: v for k, v in report['config'].items()
                    if isinstance(v, (str, int, float, bool))})

eval_by_round = {
    er.get('round_num', 0): er.get('accuracy', 0.0)
    for er in report.get('eval_results', [])
    if isinstance(er, dict)
}

for rm in report['round_metrics']:
    rn = rm['round']
    acc.add_eval_result(EvalResult(
        run_label            = 'MedTrace FL',
        round_num            = rn,
        accuracy             = eval_by_round.get(rn, 0.0),
        loss                 = rm['avg_loss'],
        perplexity           = 0.0,
        num_eval_samples     = rm['total_samples'],
        elapsed_seconds      = rm['aggregation_time'],
        privacy_budget_spent = 0.0,
    ))
    for hid, m in rm.get('hospital_metrics', {}).items():
        acc.add_client_metrics(hid, m)

# ── Render and display ────────────────────────────────────────────────────
plot_dir = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(plot_dir, exist_ok=True)
plotter  = ResultsPlotter(output_dir=plot_dir, fmt='png', dpi=120)
paths    = plotter.save_all(acc)

for path in paths:
    img = plt.imread(path)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Saved: {path}')

results_path = os.path.join(OUTPUT_DIR, 'results.json')
acc.save_json(results_path)
print(f'\nResults JSON: {results_path}')
print(f'Download all outputs from the Output tab on the right sidebar.')


## Step 6 — Evaluate the trained model

Loads the final federated global model and runs inference on 3 clinical questions.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ── Locate the final global model ────────────────────────────────────────
_target = os.path.join(cfg.global_model_dir, f'round_{cfg.fl_rounds - 1}')
if not os.path.exists(_target):
    _available = sorted(
        int(d.split('_')[1])
        for d in os.listdir(cfg.global_model_dir)
        if d.startswith('round_')
        and os.path.isdir(os.path.join(cfg.global_model_dir, d))
    )
    if not _available:
        raise FileNotFoundError(
            f'No model rounds found in {cfg.global_model_dir}. '
            'Did training complete?'
        )
    _target = os.path.join(cfg.global_model_dir, f'round_{_available[-1]}')
    print(f'Note: loading round {_available[-1]} (highest available)')

print(f'Loading model from: {_target}')
_device    = 'cuda' if torch.cuda.is_available() else 'cpu'
_tokenizer = AutoTokenizer.from_pretrained(_target)
_base      = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    torch_dtype = torch.float16,
    device_map  = 'auto',
)
_model = PeftModel.from_pretrained(_base, _target)
_model.eval()
print('Model ready.\n')

# ── Inference helper ─────────────────────────────────────────────────────
def ask(question: str, max_new_tokens: int = 400) -> str:
    prompt = (
        f'<|system|>\n{cfg.system_msg}</s>\n'
        f'<|user|>\n{question}</s>\n'
        f'<|assistant|>\n'
    )
    inputs = _tokenizer(prompt, return_tensors='pt').to(_device)
    with torch.no_grad():
        out = _model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.7,
            do_sample          = True,
            repetition_penalty = 1.1,
            pad_token_id       = _tokenizer.eos_token_id,
        )
    return _tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    )

# ── Clinical evaluation questions ────────────────────────────────────────
_eval_questions = [
    'A 62-year-old man with hypertension has crushing chest pain radiating '
    'to the left arm, diaphoresis, and nausea for 45 min. ECG shows ST '
    'elevation in II, III, aVF. What is the immediate management?',

    'A 55-year-old woman has sudden left-sided weakness, facial droop, and '
    'slurred speech for 2 hours. CT head shows no hemorrhage. Next step?',

    'A returned traveller from sub-Saharan Africa has cyclic fever, rigors, '
    'and splenomegaly for 5 days. Blood smear shows ring-form trophozoites. '
    'Treatment?',
]

for i, q in enumerate(_eval_questions, 1):
    print(f'Q{i}: {q}')
    print(f'A:  {ask(q)[:600]}')
    print('─' * 70)

# ── Free GPU memory when done ─────────────────────────────────────────────
del _model, _base, _tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Memory freed.')
